# 04 — Full model fine-tuning

**Purpose:** Update all model parameters using the same examples, chat loss, optimizer family, and evaluation rules as LoRA.

**Platform:** Kaggle or Colab (Python + GPU for model stages). Never use held-out test data for model selection.

**Start here:** The first code cell clones the public GitHub repository automatically when needed. Enable Internet in Kaggle/Colab.

In [ ]:
# Kaggle/Colab project setup — NO PIP INSTALLATIONS.
# Uses the versions already installed in this runtime.
from pathlib import Path
from importlib import metadata
import os, sys, subprocess

REPO_URL = "https://github.com/muhammad-omer-dev/nlp_research_proj.git"
PULL_LATEST = False  # Enable only for a clean clone after pushing fixes to GitHub.
if Path("/kaggle/working").is_dir():
    WORKSPACE = Path("/kaggle/working")
elif Path("/content").is_dir():
    WORKSPACE = Path("/content")
else:
    WORKSPACE = Path.cwd()
REPO_DIR = WORKSPACE / "nlp_research_proj"
ROOT = REPO_DIR / "Urdu_Qwen_Research_Notebooks"

if (REPO_DIR / ".git").is_dir():
    if PULL_LATEST:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
elif REPO_DIR.exists():
    raise RuntimeError(f"Existing non-git folder: {REPO_DIR}; inspect before removing it")
elif (Path.cwd() / "project_utils.py").is_file():
    ROOT = Path.cwd()
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

if not (ROOT / "project_utils.py").is_file():
    raise FileNotFoundError(f"Project code missing from {ROOT}")

# Check user-confirmed Kaggle versions; NEVER uninstall or upgrade preinstalled dependencies.
EXPECTED = {
    "transformers": "5.16.1",
    "peft": "0.20.0",
    "torch": "2.11.0",  # the Kaggle build has a +cu128 suffix
    "numpy": "2.1.3",
    "scipy": "1.16.3",
    "scikit-learn": "1.6.1",
}
versions = {}
for name, expected in EXPECTED.items():
    try:
        got = metadata.version(name)
    except metadata.PackageNotFoundError:
        raise RuntimeError(f"Missing preinstalled dependency: {name}") from None
    versions[name] = got
    if got.split("+")[0] != expected:
        raise RuntimeError(
            f"This code is checked for {name}={expected}, found {got}. "
            "Do NOT run pip; use the tested Kaggle image or review the API changes."
        )
for name in ("accelerate", "datasets", "pandas", "matplotlib", "pytest"):
    try:
        versions[name] = metadata.version(name)
    except metadata.PackageNotFoundError:
        raise RuntimeError(f"Required package missing: {name}. This runtime cannot run the selected notebook.") from None
print("Using preinstalled Kaggle libraries (no installations):", versions)

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from project_utils import *
folders()
api = check_training_api()
print("Project ready:", ROOT)
print("Training API ready:", api["transformers"], api["trainer_tokenizer_key"], api["warmup_key"])


**IMPORTANT MEMORY NOTE:** Full training of 0.5B requires model weights, gradients, and optimizer state. Adafactor is selected **for both LoRA and full** to reduce optimizer-memory pressure and keep the optimizer family controlled. It is *not* guaranteed to fit T4 in every session. Start with the 2-step pilot in notebook 00 and batch size 1.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable a GPU'
TASK='sib200'; LANG='ur'; SIZES=[10]; SEEDS=[42]
MODEL_OVERRIDE=None # optionally an English-trained merged checkpoint, same as LoRA notebook
DRY_RUN=False
for seed in SEEDS:
    for n in SIZES:
        run_name=f'{TASK}_full_{LANG}_n{n}_s{seed}' + ('_anchor' if MODEL_OVERRIDE else '')
        if (ROOT/'outputs/checkpoints'/run_name/'complete.json').exists():
            print('Already complete:',run_name);continue
        if DRY_RUN: print('Would run',run_name); continue
        try:
            r=train_run('full',TASK,n,seed,run_name=run_name,lang=LANG,model_override=MODEL_OVERRIDE)
            print('FINISHED',r['run_id'],'minutes:',round(r['wall_seconds']/60,2),'GPU GiB:',r['peak_gpu_allocated_gib'])
        except torch.cuda.OutOfMemoryError as exc:
            torch.cuda.empty_cache()
            print('FULL TRAIN OOM. Stop this kernel; reduce max_length to 128 or use more memory.')
            print('Do not claim a full-tuning result until a completed checkpoint is saved.')
            raise

When you have successful results at 10/class, add 25 and 50/class. Compare the same sizes and seeds as LoRA. Learning rates can differ between methods, but tune them fairly using the validation set, then freeze settings before final test evaluation.